# Amazon Bedrock AgentCore Policy - Lambda 타겟 정리

## 개요

이 노트북은 `01-Setup-Gateway-Lambda.ipynb`와 `02-Policy-Enforcement.ipynb`에서 생성한 모든 AWS 리소스를 삭제합니다.

### 삭제할 리소스

| 순서 | 리소스 | 설명 |
|------|--------|------|
| 1 | Policy Engine | Cedar 정책 엔진 (Gateway에서 분리 후 삭제) |
| 2 | Gateway Target | Gateway에 연결된 Lambda 타겟 |
| 3 | AgentCore Gateway | MCP 프로토콜 엔드포인트 |
| 4 | Custom Claims Lambda | Cognito Pre Token Generation 트리거용 Lambda |
| 5 | Refund Lambda | 샘플 환불 처리 Lambda 함수 |
| 6 | Cognito App Client | OAuth2 클라이언트 |
| 7 | Cognito Resource Server | OAuth2 scope 정의 |
| 8 | Cognito Domain | OAuth2 토큰 엔드포인트 도메인 |
| 9 | Cognito User Pool | 인증 컨테이너 |
| 10 | Config Files | 설정 파일 (선택) |

> **주의**: 이 노트북을 실행하면 모든 리소스가 **영구적으로 삭제**됩니다.

---

## Part 1: 환경 설정

In [ ]:
import json
import sys
import time
from pathlib import Path

import boto3
from botocore.exceptions import ClientError

print("✓ Libraries loaded")

### Step 1.1: 설정 파일 로드

In [ ]:
# Load configuration file
config_path = Path.cwd() / "gateway_config.json"

if not config_path.exists():
    print("⚠️  gateway_config.json not found.")
    print("   Resources may have already been cleaned up, or setup was not run.")
    CONFIG = None
else:
    with open(config_path, "r") as f:
        CONFIG = json.load(f)
    
    print("✓ Configuration loaded\n")
    
    # Extract values
    region = CONFIG.get("region")
    gateway_id = CONFIG.get("gateway_id")
    gateway_arn = CONFIG.get("gateway_arn")
    policy_engine_id = CONFIG.get("policy_engine_id")
    lambda_arn = CONFIG.get("lambda_arn")
    client_info = CONFIG.get("client_info", {})
    user_pool_id = client_info.get("user_pool_id")
    client_id = client_info.get("client_id")
    domain_prefix = client_info.get("domain_prefix")
    scope = client_info.get("scope", "")
    resource_server_id = scope.split("/")[0] if scope else "TestGateway"
    custom_claims_lambda = f"cognito-custom-claims-{user_pool_id}" if user_pool_id else None
    
    print("=" * 70)
    print("🗑️  삭제될 리소스 목록")
    print("=" * 70)
    
    print("\n📦 Policy Engine:")
    print(f"   - Policy Engine ID: {policy_engine_id or 'Not set'}")
    
    print("\n🌐 Gateway:")
    print(f"   - Gateway ID: {gateway_id}")
    print(f"   - Gateway ARN: {gateway_arn}")
    
    print("\n⚡ Lambda Functions:")
    print(f"   - Refund Lambda: {lambda_arn.split(':')[-1] if lambda_arn else 'RefundLambda'}")
    print(f"   - Custom Claims Lambda: {custom_claims_lambda or 'Not set'}")
    
    print("\n🔐 Cognito:")
    print(f"   - User Pool ID: {user_pool_id}")
    print(f"   - App Client ID: {client_id}")
    print(f"   - Resource Server: {resource_server_id}")
    print(f"   - Domain: {domain_prefix}")
    
    print("\n📄 Config Files:")
    print(f"   - gateway_config.json")
    
    print("\n" + "=" * 70)
    print(f"Region: {region}")
    print("=" * 70)

### Step 1.2: AWS 클라이언트 초기화

In [ ]:
if CONFIG:
    REGION = CONFIG["region"]
    session = boto3.Session(region_name=REGION)
    
    sts_client = session.client("sts")
    lambda_client = session.client("lambda")
    cognito_client = session.client("cognito-idp")
    policy_client = session.client("bedrock-agentcore-control")
    gateway_client = session.client("bedrock-agentcore-control")
    
    ACCOUNT_ID = sts_client.get_caller_identity()["Account"]
    
    print("✓ AWS clients initialized")
    print(f"  Account ID: {ACCOUNT_ID}")
    print(f"  Region: {REGION}")
else:
    print("⚠️  Skipping - no configuration loaded")

---

## Part 2: Policy Engine 정리

Policy Engine을 Gateway에서 분리하고 삭제합니다.

### Step 2.1: Gateway에서 Policy Engine 분리

In [ ]:
if CONFIG:
    GATEWAY_ID = CONFIG.get("gateway_id")
    POLICY_ENGINE_ID = CONFIG.get("policy_engine_id")
    
    if POLICY_ENGINE_ID and GATEWAY_ID:
        print("=" * 70)
        print("Step 2.1: Detaching Policy Engine from Gateway")
        print("=" * 70)
        
        try:
            # Get current gateway configuration
            gateway = gateway_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
            
            # Check if policy engine is attached
            policy_config = gateway.get("authorizationConfig", {}).get("cedarPoliciesConfiguration")
            
            if policy_config:
                # Update gateway to remove policy engine
                gateway_client.update_gateway(
                    gatewayIdentifier=GATEWAY_ID,
                    authorizationConfig={
                        "cedarPoliciesConfiguration": {}
                    }
                )
                print(f"✓ Policy Engine detached from Gateway")
                
                # Wait for gateway to be ready
                print("  Waiting for Gateway to be ready...")
                time.sleep(5)
            else:
                print("  Policy Engine was not attached to Gateway")
                
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                print(f"  Gateway {GATEWAY_ID} not found (may already be deleted)")
            else:
                print(f"  Error: {e}")
    else:
        print("⚠️  No Policy Engine ID found in config")
else:
    print("⚠️  Skipping - no configuration loaded")

### Step 2.2: Policy Engine 내 정책 삭제

In [ ]:
if CONFIG and POLICY_ENGINE_ID:
    print("=" * 70)
    print("Step 2.2: Deleting all policies in Policy Engine")
    print("=" * 70)
    
    try:
        # List all policies
        paginator = policy_client.get_paginator("list_policies")
        deleted_count = 0
        
        for page in paginator.paginate(policyEngineId=POLICY_ENGINE_ID):
            for policy in page.get("policies", []):
                policy_id = policy["policyId"]
                try:
                    policy_client.delete_policy(
                        policyEngineId=POLICY_ENGINE_ID,
                        policyId=policy_id
                    )
                    print(f"  ✓ Deleted policy: {policy_id}")
                    deleted_count += 1
                except ClientError as e:
                    print(f"  ✗ Failed to delete policy {policy_id}: {e}")
        
        if deleted_count == 0:
            print("  No policies found")
        else:
            print(f"\n✓ Deleted {deleted_count} policies")
            
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Policy Engine {POLICY_ENGINE_ID} not found")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  Skipping - no Policy Engine ID")

### Step 2.3: Policy Engine 삭제

In [ ]:
if CONFIG and POLICY_ENGINE_ID:
    print("=" * 70)
    print("Step 2.3: Deleting Policy Engine")
    print("=" * 70)
    
    try:
        policy_client.delete_policy_engine(
            policyEngineId=POLICY_ENGINE_ID
        )
        print(f"✓ Policy Engine deleted: {POLICY_ENGINE_ID}")
        
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"  Policy Engine {POLICY_ENGINE_ID} not found (may already be deleted)")
        else:
            print(f"  Error: {e}")
else:
    print("⚠️  Skipping - no Policy Engine ID")

---

## Part 3: Gateway 정리

Gateway Target을 삭제한 후 Gateway를 삭제합니다.

### Step 3.1: Gateway Target 삭제

In [ ]:
if CONFIG:
    GATEWAY_ID = CONFIG.get("gateway_id")
    
    if GATEWAY_ID:
        print("=" * 70)
        print("Step 3.1: Deleting Gateway Targets")
        print("=" * 70)
        
        try:
            # List all targets
            targets = gateway_client.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)
            
            for target in targets.get("items", []):
                target_id = target["targetId"]
                target_name = target.get("name", "Unknown")
                
                try:
                    gateway_client.delete_gateway_target(
                        gatewayIdentifier=GATEWAY_ID,
                        targetId=target_id
                    )
                    print(f"  ✓ Deleted target: {target_name} ({target_id})")
                    
                    # Wait for target deletion
                    time.sleep(2)
                    
                except ClientError as e:
                    print(f"  ✗ Failed to delete target {target_name}: {e}")
            
            if not targets.get("items"):
                print("  No targets found")
                
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                print(f"  Gateway {GATEWAY_ID} not found")
            else:
                print(f"  Error: {e}")
    else:
        print("⚠️  No Gateway ID found in config")
else:
    print("⚠️  Skipping - no configuration loaded")

### Step 3.2: Gateway 삭제

In [ ]:
if CONFIG:
    GATEWAY_ID = CONFIG.get("gateway_id")
    
    if GATEWAY_ID:
        print("=" * 70)
        print("Step 3.2: Deleting Gateway")
        print("=" * 70)
        
        try:
            gateway_client.delete_gateway(gatewayIdentifier=GATEWAY_ID)
            print(f"✓ Gateway deletion initiated: {GATEWAY_ID}")
            
            # Wait for gateway deletion
            print("  Waiting for Gateway to be deleted...")
            for i in range(30):
                try:
                    gateway = gateway_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
                    status = gateway.get("status", "UNKNOWN")
                    print(f"  Status: {status}")
                    time.sleep(5)
                except ClientError as e:
                    if e.response["Error"]["Code"] == "ResourceNotFoundException":
                        print("✓ Gateway deleted successfully")
                        break
                    else:
                        raise
            else:
                print("⚠️  Gateway deletion is taking longer than expected")
                
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                print(f"  Gateway {GATEWAY_ID} not found (may already be deleted)")
            else:
                print(f"  Error: {e}")
    else:
        print("⚠️  No Gateway ID found in config")
else:
    print("⚠️  Skipping - no configuration loaded")

---

## Part 4: Lambda 함수 정리

Cognito Lambda Trigger와 Lambda 함수들을 삭제합니다.

### Step 4.1: Cognito Lambda Trigger 제거

In [ ]:
if CONFIG:
    USER_POOL_ID = CONFIG.get("client_info", {}).get("user_pool_id")
    
    if USER_POOL_ID:
        print("=" * 70)
        print("Step 4.1: Removing Cognito Lambda Trigger")
        print("=" * 70)
        
        try:
            # Get current User Pool configuration
            user_pool = cognito_client.describe_user_pool(UserPoolId=USER_POOL_ID)
            lambda_config = user_pool.get("UserPool", {}).get("LambdaConfig", {})
            
            if lambda_config.get("PreTokenGenerationConfig") or lambda_config.get("PreTokenGeneration"):
                # Remove Lambda trigger
                cognito_client.update_user_pool(
                    UserPoolId=USER_POOL_ID,
                    LambdaConfig={}
                )
                print("✓ Lambda trigger removed from User Pool")
            else:
                print("  No Lambda trigger configured")
                
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                print(f"  User Pool {USER_POOL_ID} not found")
            else:
                print(f"  Error: {e}")
    else:
        print("⚠️  No User Pool ID found in config")
else:
    print("⚠️  Skipping - no configuration loaded")

### Step 4.2: Custom Claims Lambda 함수 삭제

In [ ]:
if CONFIG:
    USER_POOL_ID = CONFIG.get("client_info", {}).get("user_pool_id")
    
    if USER_POOL_ID:
        print("=" * 70)
        print("Step 4.2: Deleting Custom Claims Lambda Function")
        print("=" * 70)
        
        # Lambda function name follows pattern: cognito-custom-claims-{user_pool_id}
        custom_claims_lambda = f"cognito-custom-claims-{USER_POOL_ID}"
        
        try:
            lambda_client.delete_function(FunctionName=custom_claims_lambda)
            print(f"✓ Deleted Lambda function: {custom_claims_lambda}")
            
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                print(f"  Lambda function {custom_claims_lambda} not found (may not have been created)")
            else:
                print(f"  Error: {e}")
    else:
        print("⚠️  No User Pool ID found in config")
else:
    print("⚠️  Skipping - no configuration loaded")

### Step 4.3: Refund Lambda 함수 삭제

In [ ]:
if CONFIG:
    LAMBDA_ARN = CONFIG.get("lambda_arn")
    
    print("=" * 70)
    print("Step 4.3: Deleting Refund Lambda Function")
    print("=" * 70)
    
    if LAMBDA_ARN:
        lambda_name = LAMBDA_ARN.split(":")[-1]
        
        try:
            lambda_client.delete_function(FunctionName=lambda_name)
            print(f"✓ Deleted Lambda function: {lambda_name}")
            
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                print(f"  Lambda function {lambda_name} not found (may already be deleted)")
            else:
                print(f"  Error: {e}")
    else:
        # Try default name
        try:
            lambda_client.delete_function(FunctionName="RefundLambda")
            print("✓ Deleted Lambda function: RefundLambda")
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                print("  Lambda function RefundLambda not found")
            else:
                print(f"  Error: {e}")
else:
    print("⚠️  Skipping - no configuration loaded")

---

## Part 5: Cognito 정리

Cognito 리소스들을 순서대로 삭제합니다.

### Step 5.1: Cognito App Client 삭제

In [ ]:
if CONFIG:
    USER_POOL_ID = CONFIG.get("client_info", {}).get("user_pool_id")
    CLIENT_ID = CONFIG.get("client_info", {}).get("client_id")
    
    if USER_POOL_ID and CLIENT_ID:
        print("=" * 70)
        print("Step 5.1: Deleting Cognito App Client")
        print("=" * 70)
        
        try:
            cognito_client.delete_user_pool_client(
                UserPoolId=USER_POOL_ID,
                ClientId=CLIENT_ID
            )
            print(f"✓ Deleted App Client: {CLIENT_ID}")
            
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                print(f"  App Client {CLIENT_ID} not found")
            else:
                print(f"  Error: {e}")
    else:
        print("⚠️  Missing User Pool ID or Client ID")
else:
    print("⚠️  Skipping - no configuration loaded")

### Step 5.2: Cognito Resource Server 삭제

In [ ]:
if CONFIG:
    USER_POOL_ID = CONFIG.get("client_info", {}).get("user_pool_id")
    SCOPE = CONFIG.get("client_info", {}).get("scope", "")
    
    if USER_POOL_ID:
        print("=" * 70)
        print("Step 5.2: Deleting Cognito Resource Server")
        print("=" * 70)
        
        # Extract resource server identifier from scope (format: {identifier}/{scope_name})
        resource_server_id = SCOPE.split("/")[0] if SCOPE else "TestGateway"
        
        try:
            cognito_client.delete_resource_server(
                UserPoolId=USER_POOL_ID,
                Identifier=resource_server_id
            )
            print(f"✓ Deleted Resource Server: {resource_server_id}")
            
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                print(f"  Resource Server {resource_server_id} not found")
            else:
                print(f"  Error: {e}")
    else:
        print("⚠️  No User Pool ID found in config")
else:
    print("⚠️  Skipping - no configuration loaded")

### Step 5.3: Cognito Domain 삭제

In [ ]:
if CONFIG:
    USER_POOL_ID = CONFIG.get("client_info", {}).get("user_pool_id")
    DOMAIN_PREFIX = CONFIG.get("client_info", {}).get("domain_prefix")
    
    if USER_POOL_ID and DOMAIN_PREFIX:
        print("=" * 70)
        print("Step 5.3: Deleting Cognito Domain")
        print("=" * 70)
        
        try:
            cognito_client.delete_user_pool_domain(
                Domain=DOMAIN_PREFIX,
                UserPoolId=USER_POOL_ID
            )
            print(f"✓ Deleted Domain: {DOMAIN_PREFIX}")
            
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                print(f"  Domain {DOMAIN_PREFIX} not found")
            else:
                print(f"  Error: {e}")
    else:
        print("⚠️  Missing User Pool ID or Domain Prefix")
else:
    print("⚠️  Skipping - no configuration loaded")

### Step 5.4: Cognito User Pool 삭제

In [ ]:
if CONFIG:
    USER_POOL_ID = CONFIG.get("client_info", {}).get("user_pool_id")
    
    if USER_POOL_ID:
        print("=" * 70)
        print("Step 5.4: Deleting Cognito User Pool")
        print("=" * 70)
        
        try:
            cognito_client.delete_user_pool(UserPoolId=USER_POOL_ID)
            print(f"✓ Deleted User Pool: {USER_POOL_ID}")
            
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                print(f"  User Pool {USER_POOL_ID} not found")
            else:
                print(f"  Error: {e}")
    else:
        print("⚠️  No User Pool ID found in config")
else:
    print("⚠️  Skipping - no configuration loaded")

---

## Part 6: 설정 파일 정리

### Step 6.1: 설정 파일 삭제 (선택)

In [ ]:
print("=" * 70)
print("Step 6.1: Deleting Configuration Files")
print("=" * 70)

config_files = [
    Path.cwd() / "gateway_config.json",
]

for config_file in config_files:
    if config_file.exists():
        config_file.unlink()
        print(f"✓ Deleted: {config_file.name}")
    else:
        print(f"  {config_file.name} not found")

print("\n✓ Configuration files cleaned up")

---

## 결론

### 삭제된 리소스

✅ Policy Engine 및 정책  
✅ Gateway Target  
✅ AgentCore Gateway  
✅ Custom Claims Lambda 함수  
✅ Refund Lambda 함수  
✅ Cognito App Client  
✅ Cognito Resource Server  
✅ Cognito Domain  
✅ Cognito User Pool  
✅ 설정 파일  

### 다시 시작하려면

`01-Setup-Gateway-Lambda.ipynb`를 다시 실행하여 모든 리소스를 재생성하세요.